In [2]:
import torch
from PIL import Image
from transformers import XLMRobertaTokenizer
from torchvision import transforms
from tqdm import tqdm
import pandas as pd
import os
import json
from collections import Counter

os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # 사용할 GPU 설정

# --- 1. 기본 설정 및 모델 구조 import ---
# BEiT-3 레포지토리의 모델링 파일을 import해야 합니다.
# 이 파일이 있는 경로를 sys.path에 추가하거나, 같은 디렉토리에 복사해두세요.
try:
    from modeling_finetune import beit3_large_patch16_768_vqav2
except ImportError:
    print("Error: 'modeling_finetune.py'를 찾을 수 없습니다.")
    print("BEiT-3 레포지토리에서 해당 파일을 다운로드하여 같은 디렉토리에 위치시키거나, sys.path에 경로를 추가해주세요.")
    exit()

In [7]:
# --- 2. 경로 설정 (사용자 환경에 맞게 수정) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fine-tuning된 모델 가중치 파일 경로
model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3_large_indomain_patch16_768_vgqaaug_vqa.pth"
# 문장 토크나이저 모델 경로
spm_model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3.spm"
# VQAv2 답변 라벨 파일 경로
vqa_answer_label_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/v2_mscoco_train2014_annotations.json"
# 캡션이 포함된 CSV 파일 경로 (이전 LLM 단계에서 생성한 파일)
# 이 파일에는 'ID', 'img_path', 'Question', 'A', 'B', 'C', 'D', 'caption_A', 'caption_B', 'caption_C', 'caption_D' 열이 있어야 합니다.
caption_csv_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/test_with_captions.csv'
# 원본 이미지 디렉토리 경로
image_dir = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/'
# 제출 샘플 파일 경로
submission_csv_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/submission/test_finetuning_llm_2.csv'
# 최종 결과 저장 경로
output_csv_path = './submission_beit3_with_captions.csv'


In [8]:
# --- 3. 모델 및 토크나이저 불러오기 ---
print("모델과 토크나이저를 로딩합니다...")
tokenizer = XLMRobertaTokenizer(spm_model_path)
model = beit3_large_patch16_768_vqav2()

checkpoint = torch.load(model_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model.to(device)
model.eval()
print("모델 로딩 완료.")

모델과 토크나이저를 로딩합니다...
모델 로딩 완료.


In [9]:
# --- 4. VQAv2 답변 사전 생성 ---
print("VQAv2 답변 사전을 생성합니다...")
try:
    with open(vqa_answer_label_path, 'r') as f:
        vqa_data = json.load(f)
    raw_annotations = vqa_data['annotations']
    answer_counter = Counter(ann['multiple_choice_answer'] for ann in raw_annotations)
    num_top_answers = 3129
    top_answers = answer_counter.most_common(num_top_answers)
    answer_vocab = [answer for answer, count in top_answers]
    answer_map = {str(i): answer for i, answer in enumerate(answer_vocab)}
    answer_to_idx = {answer: i for i, answer in enumerate(answer_vocab)}

    # 'yes'의 인덱스를 미리 찾아둡니다. 이 로직의 핵심입니다.
    if 'yes' in answer_to_idx:
        yes_idx = answer_to_idx['yes']
        print(f"답변 사전 생성 완료. 'yes'의 인덱스는 {yes_idx} 입니다.")
    else:
        raise ValueError("'yes'를 답변 사전에서 찾을 수 없습니다!")

except (FileNotFoundError, KeyError, ValueError) as e:
    print(f"답변 사전 생성 중 오류 발생: {e}")
    exit()

VQAv2 답변 사전을 생성합니다...
답변 사전 생성 완료. 'yes'의 인덱스는 0 입니다.


In [14]:
count=0

# --- 5. 데이터 로드 및 추론 준비 ---
print(f"캡션 데이터를 '{caption_csv_path}'에서 로드합니다.")
test_df = pd.read_csv(caption_csv_path)
results = []

# 이미지 전처리(Transform) 정의
transform = transforms.Compose([
    transforms.Resize((768, 768)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])


# --- 6. 캡션을 이용한 추론 실행 ---
print("캡션을 질문으로 사용하여 'yes' 점수 기반 추론을 시작합니다...")
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    image_path = os.path.join(image_dir, row['img_path'].lstrip('./'))

    # 이미지 유효성 검사 및 전처리
    if not os.path.exists(image_path):
        results.append({'ID': row['ID'], 'answer': '?'}) # 제출 형식에 맞게 ID와 answer 저장
        continue
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
    except Exception as e:
        print(f"이미지 처리 오류 {image_path}: {e}")
        results.append({'ID': row['ID'], 'answer': '?'})
        continue

    # 각 캡션에 대한 'yes' 점수를 저장할 딕셔너리
    choice_scores = {}
    
    # 4개의 캡션을 하나씩 모델의 질문으로 사용
    for choice_key in ['A', 'B', 'C', 'D']:
        caption = row[f'caption_{choice_key}']
        
        # 캡션이 유효한지 확인 (LLM이 생성을 못했을 경우 대비)
        if not isinstance(caption, str) or not caption:
            continue

        print(f"Processing ID: {row['ID']}, Choice: {choice_key}, Caption: {caption}")
        # 캡션을 토크나이징
        encoding = tokenizer(
            caption, padding='max_length', truncation=True,
            max_length=40, return_tensors='pt'
        )
        question_tokens = encoding['input_ids'].to(device)
        padding_mask = encoding['attention_mask'].to(device)

        with torch.no_grad():
            output_logits = model(
                image=image_tensor,
                question=question_tokens,
                padding_mask=padding_mask
            )

        argmax_idx = output_logits.argmax(dim=-1)
        answer = answer_map[str(argmax_idx.item())]
        print(f"Predicted Answer for {choice_key}: {answer}")

        # 전체 답변 점수(logits)에서 'yes'에 해당하는 점수를 추출
        yes_score = output_logits[0, yes_idx].item()
        choice_scores[choice_key] = yes_score

    # 가장 높은 'yes' 점수를 받은 보기를 정답으로 선택
    if choice_scores:
        best_choice_key = max(choice_scores, key=choice_scores.get)
        predicted_answer = row[best_choice_key]
    else:
        predicted_answer = row['A'] # 모든 캡션이 비정상일 경우 기본값 'A' 선택

    results.append({'ID': row['ID'], 'answer': predicted_answer})

    if count <1:
        print(f"ID: {row['ID']}, Predicted Answer: {predicted_answer}")
        count += 1
    else:
        break


# --- 7. 최종 결과 제출 파일로 저장 ---
submission_df = pd.DataFrame(results)

# 원본 제출 파일 형식에 맞게 ID 순으로 정렬 (선택사항이지만 권장)
try:
    sample_submission_df = pd.read_csv(submission_csv_path)
    # ID 순서를 샘플 제출 파일에 맞춤
    submission_df = submission_df.set_index('ID').loc[sample_submission_df['ID']].reset_index()
except FileNotFoundError:
    print(f"Warning: '{submission_csv_path}'를 찾을 수 없어 ID 순서 정렬을 건너뜁니다.")


# json없을 경우 생성
if not os.path.exists(os.path.dirname(output_csv_path)):
    os.makedirs(os.path.dirname(output_csv_path))

submission_df.to_csv(output_csv_path, index=False)

print(f"\n✅ 추론 완료! 최종 결과가 '{output_csv_path}' 파일에 저장되었습니다.")
print("\n--- 결과 미리보기 (상위 5개) ---")
print(submission_df.head())

캡션 데이터를 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/test_with_captions.csv'에서 로드합니다.
캡션을 질문으로 사용하여 'yes' 점수 기반 추론을 시작합니다...


  0%|          | 0/680 [00:00<?, ?it/s]

Processing ID: TEST_000, Choice: A, Caption: A photo of a pizza.
Predicted Answer for A: 20
Processing ID: TEST_000, Choice: B, Caption: A photo of blueberries.
Predicted Answer for B: 20
Processing ID: TEST_000, Choice: C, Caption: A photo of salmon.
Predicted Answer for C: 20
Processing ID: TEST_000, Choice: D, Caption: A photo of an avocado.


  0%|          | 1/680 [00:00<07:00,  1.61it/s]

Predicted Answer for D: 20
ID: TEST_000, Predicted Answer: Blueberries
Processing ID: TEST_001, Choice: A, Caption: A person building muscle and strength.
Predicted Answer for A: green
Processing ID: TEST_001, Choice: B, Caption: A person practicing for a marathon.
Predicted Answer for B: green
Processing ID: TEST_001, Choice: C, Caption: A person training for a cycling race.
Predicted Answer for C: green
Processing ID: TEST_001, Choice: D, Caption: A person preparing for a swimming competition.


  0%|          | 1/680 [00:01<13:50,  1.22s/it]

Predicted Answer for D: green

✅ 추론 완료! 최종 결과가 './submission_beit3_with_captions.csv' 파일에 저장되었습니다.

--- 결과 미리보기 (상위 5개) ---
         ID                        answer
0  TEST_000                   Blueberries
1  TEST_001  Building muscle and strength


In [15]:
submission_df

,ID,answer
0,TEST_000,Blueberries
1,TEST_001,Building muscle and strength
2,TEST_002,To a school
3,TEST_003,In a city park
4,TEST_004,Doing laundry
...,...,...
675,TEST_675,They are coworkers
676,TEST_676,Night
677,TEST_677,Glasses
678,TEST_678,Wagyu


In [ ]:
# --- 7. 최종 결과 제출 파일로 저장 (내용 -> A/B/C/D 변환 로직 포함) ---
print("추론 결과를 A, B, C, D 문자로 변환하여 저장합니다.")

merged_df = pd.merge(submission_df, test_df[['ID', 'A', 'B', 'C', 'D']], on='ID', how='left')

# 7-3. 답변 내용을 A, B, C, D 문자로 변환하는 함수
def convert_text_to_choice(row):
    predicted_text = row['answer']
    
    # 각 보기의 내용과 예측된 답변 내용이 일치하는지 확인
    if predicted_text == row['A']:
        return 'A'
    elif predicted_text == row['B']:
        return 'B'
    elif predicted_text == row['C']:
        return 'C'
    elif predicted_text == row['D']:
        return 'D'
    else:
        # 만약 일치하는 보기가 없을 경우 (예: 추론 오류), 기본값으로 'A'를 반환하고 경고 메시지 출력
        print(f"Warning: ID {row['ID']}의 예측 답변 '{predicted_text}'가 A,B,C,D 보기와 일치하지 않습니다. 기본값 'A'로 처리합니다.")
        return 'A'

# 7-4. 함수를 적용하여 최종 선택지('A'/'B'/'C'/'D') 열 생성
merged_df['final_choice'] = merged_df.apply(convert_text_to_choice, axis=1)

# 7-5. 제출 형식에 맞는 최종 데이터프레임 생성 (ID와 최종 선택지만 남김)
final_submission_df = merged_df[['ID', 'final_choice']].rename(columns={'final_choice': 'answer'})

sample_submission_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv'

# 7-6. 원본 제출 파일 형식에 맞게 ID 순으로 정렬 및 저장
try:
    sample_df = pd.read_csv(sample_submission_path)
    # ID를 인덱스로 설정하여 순서를 맞춘 후, 다시 컬럼으로 리셋
    final_submission_df = final_submission_df.set_index('ID').loc[sample_df['ID']].reset_index()
except FileNotFoundError:
    print(f"Warning: '{sample_submission_path}'를 찾을 수 없어 ID 순서 정렬을 건너뜁니다.")
except Exception as e:
    print(f"ID 순서 정렬 중 오류 발생: {e}")


# 폴더가 없을 경우 생성
output_dir = os.path.dirname(submission_csv_path)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 최종 결과 저장
final_submission_df.to_csv(submission_csv_path, index=False)

print(f"\n✅ 모든 작업 완료! 최종 제출 파일이 '{submission_csv_path}'에 저장되었습니다.")
print("\n--- 제출 파일 미리보기 (상위 5개) ---")
print(final_submission_df.head())

추론 결과를 A, B, C, D 문자로 변환하여 저장합니다.
ID 순서 정렬 중 오류 발생: name 'sample_submission_path' is not defined

✅ 모든 작업 완료! 최종 제출 파일이 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv'에 저장되었습니다.

--- 제출 파일 미리보기 (상위 5개) ---
         ID answer
0  TEST_000      B
1  TEST_001      A
2  TEST_002      B
3  TEST_003      B
4  TEST_004      A
